# Gate 0: can the board's llama.cpp convert Qwen3.5 0.8B?

Before training anything, this notebook converts the **unmodified** `Qwen/Qwen3.5-0.8B` checkpoint to a Q8_0 GGUF
with `convert_hf_to_gguf.py` at the commit pinned in `config.env`, and compares the result with the GGUF the board
runs today (`bartowski/Qwen_Qwen3.5-0.8B-GGUF`). If the two files behave the same, a fine-tuned checkpoint can
take the same path.

**Kaggle settings:** Internet **on**. No GPU and no dataset needed. Run all the cells (about 15 minutes) and read
the `VERDICT` block at the end.

In [ ]:
import hashlib, json, os, subprocess, sys, time, urllib.request

WORK = "/kaggle/working"
LLAMA = f"{WORK}/llama.cpp"
LLAMA_REF = "2cbe0ec429b849a88664e0f62388fc92a5abd699"   # config.env: LLAMA_CPP_REF
BASE_REPO = "Qwen/Qwen3.5-0.8B"
REF_REPO = "bartowski/Qwen_Qwen3.5-0.8B-GGUF"            # config.env: LLM_REPO
REF_REVISION = "f36b1ea49a332ede8fe5f389bbf5b3575ef71f48"
REF_FILE = "Qwen_Qwen3.5-0.8B-Q8_0.gguf"
REF_SHA256 = "7182e2362766bb9569209bbc24cf1a4cdfbb8ab161babdb2080c84fa62c08c2f"
SERVER = f"{LLAMA}/build/bin/llama-server"
PORT = 8091
os.chdir(WORK)


def sh(cmd):
    print("$", cmd, flush=True)
    r = subprocess.run(cmd, shell=True)
    if r.returncode != 0:
        raise RuntimeError(f"command failed with exit code {r.returncode}: {cmd}")


def sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(1 << 20), b""):
            h.update(block)
    return h.hexdigest()


def post(path, payload, timeout=300):
    req = urllib.request.Request(f"http://127.0.0.1:{PORT}{path}", data=json.dumps(payload).encode(),
                                 headers={"Content-Type": "application/json"})
    with urllib.request.urlopen(req, timeout=timeout) as resp:
        return json.loads(resp.read())


def get(path, timeout=10):
    with urllib.request.urlopen(f"http://127.0.0.1:{PORT}{path}", timeout=timeout) as resp:
        return json.loads(resp.read())


class Server:
    """llama-server with the flags of board/systemd/math-llm.service."""

    def __init__(self, model):
        self.model = model

    def __enter__(self):
        self.log = open(f"{WORK}/server.log", "w")
        self.proc = subprocess.Popen(
            [SERVER, "-m", self.model, "--host", "127.0.0.1", "--port", str(PORT), "--threads", "4",
             "--ctx-size", "4096", "--parallel", "1", "--cache-ram", "0", "--ctx-checkpoints", "2",
             "--jinja", "--reasoning", "off", "--reasoning-budget", "0", "--device", "none", "-ngl", "0"],
            stdout=self.log, stderr=subprocess.STDOUT)
        for _ in range(180):
            if self.proc.poll() is not None:
                break
            try:
                if get("/health").get("status") == "ok":
                    return self
            except Exception:
                time.sleep(1)
        self.__exit__()
        print(open(f"{WORK}/server.log").read()[-3000:])
        raise RuntimeError(f"llama-server did not start with {self.model}")

    def __exit__(self, *exc):
        self.proc.terminate()
        try:
            self.proc.wait(timeout=20)
        except subprocess.TimeoutExpired:
            self.proc.kill()
        self.log.close()

In [ ]:
# llama.cpp at the commit every board runs. The build runs in the background while the next cells download.
if not os.path.exists(f"{LLAMA}/convert_hf_to_gguf.py"):
    sh(f"mkdir -p {LLAMA} && cd {LLAMA} && git init -q && "
       f"git remote add origin https://github.com/ggml-org/llama.cpp.git && "
       f"git fetch -q --depth 1 origin {LLAMA_REF} && git checkout -q FETCH_HEAD")
sh(f"cd {LLAMA} && git log -1 --format=%h")
sh("which cmake || pip install -q cmake")
build = subprocess.Popen(
    f"cd {LLAMA} && cmake -B build -DGGML_NATIVE=ON -DCMAKE_BUILD_TYPE=Release -DLLAMA_CURL=OFF > {WORK}/build.log 2>&1 && "
    f"cmake --build build --config Release -j4 --target llama-server >> {WORK}/build.log 2>&1", shell=True)

In [ ]:
# The converter needs the gguf package of the same commit, and a transformers that knows the Qwen3.5 tokenizer.
import numpy as np
sh(f"pip install -q -U 'transformers>=5' sentencepiece protobuf huggingface_hub 'numpy=={np.__version__}'")
sh(f"pip install -q --no-deps {LLAMA}/gguf-py")
import importlib.metadata as md
print({p: md.version(p) for p in ("transformers", "gguf", "torch", "numpy", "huggingface_hub")})

In [ ]:
from huggingface_hub import HfApi, hf_hub_download, snapshot_download

base_sha = HfApi().model_info(BASE_REPO).sha
print("base revision:", base_sha)
snapshot_download(BASE_REPO, revision=base_sha, local_dir=f"{WORK}/base")
sh(f"ls -la {WORK}/base")
print(json.dumps({k: v for k, v in json.load(open(f"{WORK}/base/config.json")).items()
                  if k in ("architectures", "model_type", "transformers_version")}, indent=1))

In [ ]:
# The conversion itself. --outtype q8_0 writes Q8_0 directly, so llama-quantize is not needed.
OURS = f"{WORK}/ours-q8_0.gguf"
sh(f"cd {WORK} && python {LLAMA}/convert_hf_to_gguf.py base --outtype q8_0 --outfile {OURS} 2>&1 | tail -25")
print("converted:", os.path.getsize(OURS) // (1 << 20), "MB")

In [ ]:
REF = hf_hub_download(REF_REPO, REF_FILE, revision=REF_REVISION, local_dir=f"{WORK}/ref")
ref_ok = sha256(REF) == REF_SHA256
print("reference:", os.path.getsize(REF) // (1 << 20), "MB, SHA256 matches config.env:", ref_ok)

In [ ]:
# 1. Compare what is inside the two files.
from collections import Counter
from gguf import GGUFReader


def value(field):
    try:
        return field.contents()
    except Exception:
        return None


def describe(path):
    reader = GGUFReader(path)
    kv = {name: value(field) for name, field in reader.fields.items() if not name.startswith("GGUF.")}
    tensors = {t.name: (tuple(int(x) for x in t.shape), t.tensor_type.name,
                        hashlib.sha1(t.data.tobytes()).hexdigest()) for t in reader.tensors}
    return kv, tensors


kv_ours, t_ours = describe(OURS)
kv_ref, t_ref = describe(REF)
report = {"base_revision": base_sha}

report["architecture"] = (kv_ours.get("general.architecture"), kv_ref.get("general.architecture"))
print("architecture (ours, reference):", report["architecture"])
print("tensors (ours, reference):", len(t_ours), len(t_ref))
only_ours, only_ref = sorted(set(t_ours) - set(t_ref)), sorted(set(t_ref) - set(t_ours))
print("tensor names only in ours:", only_ours[:10], "| only in the reference:", only_ref[:10])
shared = sorted(set(t_ours) & set(t_ref))
shape_diff = [n for n in shared if t_ours[n][0] != t_ref[n][0]]
type_diff = [n for n in shared if t_ours[n][1] != t_ref[n][1]]
same_bytes = sum(t_ours[n][2] == t_ref[n][2] for n in shared)
print("types in ours:     ", dict(Counter(t[1] for t in t_ours.values())))
print("types in reference:", dict(Counter(t[1] for t in t_ref.values())))
print(f"shared tensors {len(shared)}: different shape {len(shape_diff)}, different type {len(type_diff)}, "
      f"byte-identical {same_bytes}")
for n in type_diff[:12]:
    print("   type differs:", n, t_ours[n][1], "vs", t_ref[n][1])
report["tensors"] = {"ours": len(t_ours), "reference": len(t_ref), "only_ours": len(only_ours),
                     "only_reference": len(only_ref), "shape_diff": len(shape_diff),
                     "type_diff": len(type_diff), "byte_identical": same_bytes}

skip = ("general.", "tokenizer.chat_template")
kv_diff = [k for k in sorted(set(kv_ours) | set(kv_ref)) if not k.startswith(skip) and kv_ours.get(k) != kv_ref.get(k)]
print("\nmetadata keys that differ (general.* left out):", len(kv_diff))
for k in kv_diff[:25]:
    a, b = kv_ours.get(k, "<missing>"), kv_ref.get(k, "<missing>")
    if isinstance(a, list) or isinstance(b, list):
        la = len(a) if isinstance(a, list) else a
        lb = len(b) if isinstance(b, list) else b
        print(f"   {k}: lists of length {la} vs {lb}")
    else:
        print(f"   {k}: {str(a)[:70]!r} vs {str(b)[:70]!r}")
report["metadata_diff"] = kv_diff
tpl_same = kv_ours.get("tokenizer.chat_template") == kv_ref.get("tokenizer.chat_template")
print("chat template identical:", tpl_same)
report["chat_template_identical"] = tpl_same

In [ ]:
print("waiting for the llama-server build...", flush=True)
if build.wait() != 0 or not os.path.exists(SERVER):
    print(open(f"{WORK}/build.log").read()[-4000:])
    raise RuntimeError("the llama.cpp build failed")
sh(f"{SERVER} --version 2>&1 | head -2")

In [ ]:
# 2. Compare how the two files answer, with the board's server flags, temperature 0 and seed 42.
PROBLEMS = [
    "How many whole numbers from 1 to 100 are divisible by 4 but not by 8?",
    "Anna has 23 stickers. She gives 5 to each of her 3 friends. How many stickers does she have left?",
    "What is the last digit of 7 x 7 x 7 x 7 x 7?",
    "A rectangle has a perimeter of 30 cm and its length is twice its width. What is its area in square centimetres?",
    "Quants nombres de dues xifres tenen la suma de les xifres igual a 9?",
    "En Marc té 12 anys i la seva germana en té 4. D'aquí a quants anys en Marc tindrà el doble d'anys que ella?",
    "Quin és el nombre més petit més gran que 50 que és múltiple de 6 i de 8?",
    "Una granota puja 3 m cada dia i en baixa 1 cada nit. Quants dies triga a sortir d'un pou de 9 m?",
]
OPTIONS = "\nA) 9\nB) 10\nC) 12\nD) 13\nE) 15\nAnswer:"
POT_SYSTEM = ("You solve math problems by writing a short, flat Python script (no functions, no comments, no input). "
              "Always reply with a ```python code block that prints one number: the final answer.")
POT_INSTRUCTION = "\nWrite a short Python program (no def, no comments) that prints the answer."
LETTER_SYSTEM = ("You solve multiple-choice math competition problems for school students. "
                 "Read the problem and the options, then reply with the letter of the correct option.")


def ask(messages, **extra):
    r = post("/v1/chat/completions", {"messages": messages, "temperature": 0, "seed": 42, **extra})
    t = r.get("timings", {})
    return r["choices"][0]["message"]["content"], (t.get("prompt_n") or 0) + (t.get("cache_n") or 0)


def behave(model):
    rows = []
    with Server(model):
        for text in PROBLEMS:
            code, n1 = ask([{"role": "system", "content": POT_SYSTEM},
                            {"role": "user", "content": text + POT_INSTRUCTION}], max_tokens=120)
            letter, n2 = ask([{"role": "system", "content": LETTER_SYSTEM},
                              {"role": "user", "content": text + OPTIONS}], max_tokens=1, grammar="root ::= [A-E]")
            rows.append({"code": code, "code_prompt_tokens": n1, "letter": letter, "letter_prompt_tokens": n2})
    return rows


b_ours, b_ref = behave(OURS), behave(REF)
n = len(PROBLEMS)
same_tokens = sum(a["code_prompt_tokens"] == b["code_prompt_tokens"] and
                  a["letter_prompt_tokens"] == b["letter_prompt_tokens"] for a, b in zip(b_ours, b_ref))
same_code = sum(a["code"] == b["code"] for a, b in zip(b_ours, b_ref))
same_letter = sum(a["letter"] == b["letter"] for a, b in zip(b_ours, b_ref))
print(f"same prompt token counts: {same_tokens}/{n}   same program text: {same_code}/{n}   same letter: {same_letter}/{n}")
for text, a, b in zip(PROBLEMS, b_ours, b_ref):
    if a["code"] != b["code"]:
        print("\n---", text, "\nOURS:\n" + a["code"], "\nREFERENCE:\n" + b["code"])
report["behaviour"] = {"n": n, "same_prompt_tokens": same_tokens, "same_program": same_code, "same_letter": same_letter}

In [ ]:
t = report["tensors"]
checks = {
    "the reference file is the one in config.env": ref_ok,
    "same architecture": report["architecture"][0] == report["architecture"][1],
    "same tensor names and shapes": t["only_ours"] == 0 and t["only_reference"] == 0 and t["shape_diff"] == 0,
    "same chat template": report["chat_template_identical"],
    "same prompt token counts": report["behaviour"]["same_prompt_tokens"] == n,
    "same letter on at least 7 of 8": report["behaviour"]["same_letter"] >= n - 1,
    "same program text on at least 5 of 8": report["behaviour"]["same_program"] >= 5,
}
report["checks"] = checks
json.dump(report, open(f"{WORK}/gate0_report.json", "w"), indent=1)
print("=" * 60, "\nVERDICT")
for name, ok in checks.items():
    print("  PASS " if ok else "  FAIL ", name)
print(f"  tensors byte-identical: {t['byte_identical']} of {t['ours']}; tensor types that differ: {t['type_diff']}; "
      f"metadata keys that differ: {len(report['metadata_diff'])}")
print("GATE 0:", "PASS - the fine-tuned checkpoint can take this path" if all(checks.values())
      else "CHECK - copy this block and the output of the two cells above")
print("=" * 60)